# Project - QA System on Coca Cola data

## Tabel of Contents
* [Data loading](#DataLoading)
* [Embedding](#DataIngestion)
* [Indexing](#Indexing)
* [Response Synthesis](#ResponseSynthesis)
* [Query Engine](#QueryEngine)
* [Basic prompts](#BasicPrompts)
* [Sentence window retriever](#SentenceWindowRetriever)
* [Auto merging retriever](#AutoMergingRetriever)
* [Auto retriever](#AutoRetriever)
* [Router query engine](#RouterQueryEngine)
* [Sub question query engine](#SubQuestionQueryEngine)

### Setup

In [1]:
%%capture
!mkdir data
!wget "https://investors.coca-colacompany.com/filings-reports/all-sec-filings/content/0000021344-25-000011/ko-20241231.htm" -O 'data/CocaCola_annual_10K_data.pdf'


In [2]:
import os
print(os.getcwd())
os.chdir('./data')
print(os.getcwd())
os.listdir


/content
/content/data


<function posix.listdir(path=None)>

In [3]:
%%capture
!pip install llama-index-readers-file pymupdf

In [4]:
from llama_index.readers.file import PyMuPDFReader
pdf_reader = PyMuPDFReader()
documents = pdf_reader.load(file_path="/content/data/CocaCola_annual_10K_data.pdf")

In [5]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=100)
nodes = splitter.get_nodes_from_documents(documents)
len(nodes)

444

In [6]:
nodes[0]

TextNode(id_='f452f6b4-bb83-4c5a-9b61-74aebe8b2c28', embedding=None, metadata={'total_pages': 444, 'file_path': '/content/data/CocaCola_annual_10K_data.pdf', 'source': '1'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='a83b0cf7-4f82-42af-be75-cf90d6e19865', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'total_pages': 444, 'file_path': '/content/data/CocaCola_annual_10K_data.pdf', 'source': '1'}, hash='e2b4c843599f910016a690cde78d43c052ee4235f2ec7e4f6f4b8a6cc334f75a')}, metadata_template='{key}: {value}', metadata_separator='\n', text='•  [image]\n•  February 20, 2025 > Form 10-K > The Coca-Cola Company\n•  Document Links\nForm: 10-K\nAnnual report pursuant to Section 13 and 15(d)\nFebruary 20, 2025\n•  [image] Complete Filing PDF\xa0»\nIncludes filing and all documents\nDocuments\n•  10-K\xa0»\n•  EX-4.1\xa0»\n•  EX-19\xa0»\n•  EX-21.1\xa0»\n•  EX-23.1\xa0»\n•  EX-24.1\xa0»\n•  EX-31.1\xa0»\n• 

In [7]:
nodes[0].text

'•  [image]\n•  February 20, 2025 > Form 10-K > The Coca-Cola Company\n•  Document Links\nForm: 10-K\nAnnual report pursuant to Section 13 and 15(d)\nFebruary 20, 2025\n•  [image] Complete Filing PDF\xa0»\nIncludes filing and all documents\nDocuments\n•  10-K\xa0»\n•  EX-4.1\xa0»\n•  EX-19\xa0»\n•  EX-21.1\xa0»\n•  EX-23.1\xa0»\n•  EX-24.1\xa0»\n•  EX-31.1\xa0»\n•  EX-31.2\xa0»\n•  EX-32.1\xa0»\n10-K: Annual report pursuant\nto Section 13 and 15(d)\nPublished on February 20, 2025'

In [8]:
nodes[0].metadata

{'total_pages': 444,
 'file_path': '/content/data/CocaCola_annual_10K_data.pdf',
 'source': '1'}

##

In [9]:
%%capture
!pip install llama_index-embeddings-huggingface

In [10]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
BGE_embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Indexing with VectorStoreIndex

In [11]:
%%capture
!pip install llama-index-embeddings-openai

In [12]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OpenAI_API_Key')

In [13]:
from llama_index.core import VectorStoreIndex
index = VectorStoreIndex(nodes, embed_model=BGE_embed_model)
index.storage_context.persist(persist_dir="./index_storage")

In [14]:
from llama_index.core import StorageContext, load_index_from_storage

storage_context = StorageContext.from_defaults(persist_dir="./index_storage")

index = load_index_from_storage(storage_context, embed_model=BGE_embed_model)

Loading llama_index.core.storage.kvstore.simple_kvstore from ./index_storage/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from ./index_storage/index_store.json.


In [35]:
from google.colab import drive
drive.mount('/content/drive')

persist_dir = "/content/drive/MyDrive/index_storage"
index.storage_context.persist(persist_dir=persist_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [36]:
storage_context = StorageContext.from_defaults(persist_dir=persist_dir)
index = load_index_from_storage(storage_context, embed_model=BGE_embed_model)

Loading llama_index.core.storage.kvstore.simple_kvstore from /content/drive/MyDrive/index_storage/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from /content/drive/MyDrive/index_storage/index_store.json.


In [17]:
%%capture
!pip install llama-index-llms-openai

In [18]:
# configure retriever
retriever = index.as_retriever()

In [19]:
#configure response synthesizer
from llama_index.core import get_response_synthesizer

response_synthesizer = get_response_synthesizer(response_mode="compact")

In [20]:
from llama_index.core.query_engine import RetrieverQueryEngine

# assuming you have an index object already
retriever = index.as_retriever()

# create the query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer
)

# now you can query
response = query_engine.query("Tell me about Coca Cola's business")
print(response)

Coca-Cola is a total beverage company that sells its beverage products in over 200 countries and territories. The company owns or licenses various beverage brands grouped into categories such as Trademark Coca-Cola, sparkling flavors, water, sports, coffee and tea, juice, value-added dairy, plant-based beverages, and emerging beverages. Coca-Cola markets some of the world's largest nonalcoholic sparkling soft drink brands like Coca-Cola, Sprite, Coca-Cola Zero Sugar, Fanta, and Diet Coke/Coca-Cola Light. The company operates in two lines of business: concentrate operations and finished product operations. Coca-Cola's success is attributed to its ability to offer consumers a wide range of beverage options and the effectiveness of its people in executing daily operations. Additionally, Coca-Cola has entered the alcohol beverage category in various markets outside the United States and provides marketing support for other beverage brands through licenses, joint ventures, and strategic rel

In [21]:
from IPython.display import Markdown, display

In [22]:
display(Markdown(f"### Response:\n\n{response}"))

### Response:

Coca-Cola is a total beverage company that sells its beverage products in over 200 countries and territories. The company owns or licenses various beverage brands grouped into categories such as Trademark Coca-Cola, sparkling flavors, water, sports, coffee and tea, juice, value-added dairy, plant-based beverages, and emerging beverages. Coca-Cola markets some of the world's largest nonalcoholic sparkling soft drink brands like Coca-Cola, Sprite, Coca-Cola Zero Sugar, Fanta, and Diet Coke/Coca-Cola Light. The company operates in two lines of business: concentrate operations and finished product operations. Coca-Cola's success is attributed to its ability to offer consumers a wide range of beverage options and the effectiveness of its people in executing daily operations. Additionally, Coca-Cola has entered the alcohol beverage category in various markets outside the United States and provides marketing support for other beverage brands through licenses, joint ventures, and strategic relationships.

In [23]:
# now you can query
response = query_engine.query("Tell me about Coca Cola's risk")
print(response)

Coca-Cola faces risks related to unfavorable general economic and geopolitical conditions, potential challenges in successfully managing new product launches, and reliance on bottling partners for a significant portion of its business. These risks could impact the company's financial results and overall operations.


In [24]:
display(Markdown(f"### Response:\n\n{response}"))

### Response:

Coca-Cola faces risks related to unfavorable general economic and geopolitical conditions, potential challenges in successfully managing new product launches, and reliance on bottling partners for a significant portion of its business. These risks could impact the company's financial results and overall operations.

In [25]:
# now you can query
response = query_engine.query("Tell me about Coca Cola's Properties")
print(response)

Coca-Cola owns and markets a variety of beverage brands, including nonalcoholic sparkling soft drinks like Coca-Cola, Sprite, Fanta, and Diet Coke/Coca-Cola Light. They also offer water, sports, coffee, tea, juice, value-added dairy, and plant-based beverages. Additionally, Coca-Cola has entered the alcohol beverage category in markets outside the United States and has a subsidiary for alcohol products in the U.S. They focus on segments like pre-mixed cocktails, flavored alcohol beverages, and hard seltzers. Furthermore, Coca-Cola provides marketing support for other beverage brands through licenses, joint ventures, and strategic relationships.


In [26]:
display(Markdown(f"### Response:\n\n{response}"))

### Response:

Coca-Cola owns and markets a variety of beverage brands, including nonalcoholic sparkling soft drinks like Coca-Cola, Sprite, Fanta, and Diet Coke/Coca-Cola Light. They also offer water, sports, coffee, tea, juice, value-added dairy, and plant-based beverages. Additionally, Coca-Cola has entered the alcohol beverage category in markets outside the United States and has a subsidiary for alcohol products in the U.S. They focus on segments like pre-mixed cocktails, flavored alcohol beverages, and hard seltzers. Furthermore, Coca-Cola provides marketing support for other beverage brands through licenses, joint ventures, and strategic relationships.

In [27]:
from IPython.display import Markdown, display

response = query_engine.query("Tell me about Coca Cola's business model")
text = response.response if hasattr(response, "response") else str(response)

display(Markdown(f"### Response:\n\n{text}"))

### Response:

Coca-Cola operates as a total beverage company, offering a wide range of beverage products under various categories such as sparkling flavors, water, sports, coffee and tea, juice, dairy, plant-based beverages, and emerging beverages. The company markets some of the world's largest nonalcoholic sparkling soft drink brands globally. Coca-Cola distributes its branded beverage products through independent bottling partners, distributors, wholesalers, and retailers, as well as through its consolidated bottling and distribution operations. The success of Coca-Cola is attributed to its ability to provide consumers with a diverse selection of beverage options that cater to their preferences and lifestyles. Additionally, the company focuses on sustainability, investing in positive change for the world and aiming for a better shared future by improving lives across various stakeholders.

## Setup LLM

In [28]:
%%capture
!pip install llama-index-llms-openai

In [29]:
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-4o", temperature=0)

## Create different retrievers

## Sentence window retrievers

In [30]:
from llama_index.core.node_parser import SentenceWindowNodeParser
node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,
    window_metadata_key="window",
    original_text_metadata_key="original_text",
)

In [31]:
Sentence_Window_Index = VectorStoreIndex(nodes, embed_model=BGE_embed_model, show_progress=True)

Generating embeddings:   0%|          | 0/444 [00:00<?, ?it/s]

In [33]:
from google.colab import drive
drive.mount('/content/drive')

persist_dir = "/content/drive/MyDrive/index_storage"
Sentence_Window_Index.storage_context.persist(persist_dir=persist_dir)

Mounted at /content/drive


In [37]:
storage_context = StorageContext.from_defaults(persist_dir=persist_dir)
Sentence_Window_Index_StorageContext = load_index_from_storage(storage_context, embed_model=BGE_embed_model)

Loading llama_index.core.storage.kvstore.simple_kvstore from /content/drive/MyDrive/index_storage/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from /content/drive/MyDrive/index_storage/index_store.json.


In [38]:
query_engine = Sentence_Window_Index_StorageContext.as_query_engine()

In [39]:
query = "Tell me about Coca Cola's business model?"

In [40]:
response = query_engine.query(query)

In [41]:
Max_SentenceWindow_score = 0.0

In [42]:
if hasattr(response, 'source_nodes'):
    print("\nSource Nodes:")
    for i, node in enumerate(response.source_nodes):
        text = node.node.text[:500]
        score = node.score if hasattr(node, 'score') else 'N/A'
        if score > Max_SentenceWindow_score:
            Max_SentenceWindow_score = score
        print(f"Node {i+1} (Score: {score:.4f}):\n{text}\n{'-'*80}\n")


Source Nodes:
Node 1 (Score: 0.7620):
Cola; sparkling flavors; water, sports, coffee and tea; juice, value-added dairy and
plant-based beverages; and emerging beverages. We own and market several of the
world’s largest nonalcoholic sparkling soft drink brands, including Coca-Cola,
Sprite, Coca-Cola Zero Sugar, Fanta and Diet Coke/Coca-Cola Light.
We make our branded beverage products available to consumers throughout the
world through our network of independent bottling partners, distributors,
wholesalers and retailers as well as ou
--------------------------------------------------------------------------------

Node 2 (Score: 0.7511):
•Operations Review — an analysis of our consolidated results of operations
for 2024 and 2023 and year-to-year comparisons between 2024 and 2023.
An analysis of our consolidated results of operations for 2023 and 2022 and
year-to-year comparisons between 2023 and 2022 can be found in MD&A
in Part II, Item 7 of the Company’s Form 10-K for the year ended


In [43]:
print("Final Answer:")
print(response)

Final Answer:
Coca-Cola operates as a total beverage company, offering a wide range of beverage products under various categories such as sparkling flavors, water, sports, coffee and tea, juice, dairy, and plant-based beverages. The company owns and markets popular nonalcoholic sparkling soft drink brands globally. Coca-Cola distributes its branded beverages through a network of independent bottling partners, distributors, wholesalers, and retailers. The company focuses on connecting with consumers by providing a diverse range of beverage options to meet their preferences and lifestyles. Additionally, Coca-Cola emphasizes sustainability and aims to make a positive impact on the world while striving for growth and investing in improving lives.


In [44]:
from IPython.display import Markdown, display

#response = query_engine.query("Tell me about Coca Cola's business model")
text = response.response if hasattr(response, "response") else str(response)

display(Markdown(f"### Response:\n\n{text}"))

### Response:

Coca-Cola operates as a total beverage company, offering a wide range of beverage products under various categories such as sparkling flavors, water, sports, coffee and tea, juice, dairy, and plant-based beverages. The company owns and markets popular nonalcoholic sparkling soft drink brands globally. Coca-Cola distributes its branded beverages through a network of independent bottling partners, distributors, wholesalers, and retailers. The company focuses on connecting with consumers by providing a diverse range of beverage options to meet their preferences and lifestyles. Additionally, Coca-Cola emphasizes sustainability and aims to make a positive impact on the world while striving for growth and investing in improving lives.

## Automerging retrievers

In [45]:
from llama_index.core.node_parser import HierarchicalNodeParser, SentenceSplitter

In [46]:
Hierarchical_Node_Parser = HierarchicalNodeParser.from_defaults()

In [47]:
AutoMerging_nodes = Hierarchical_Node_Parser.get_nodes_from_documents(documents)

In [48]:
len(AutoMerging_nodes)

2937

In [49]:
from llama_index.core.node_parser import get_leaf_nodes, get_root_nodes

In [50]:
leaf_nodes = get_leaf_nodes(AutoMerging_nodes)

In [51]:
len(leaf_nodes)

1908

In [52]:
root_nodes = get_root_nodes(AutoMerging_nodes)

In [53]:
len(root_nodes)

444

In [54]:
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.storage import StorageContext
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

docstore = SimpleDocumentStore()

# insert nodes into docstore
docstore.add_documents(AutoMerging_nodes)

# define storage context (will include vector store by default too)
storage_context = StorageContext.from_defaults(docstore=docstore)

In [55]:
## Load index into vector index
from llama_index.core import VectorStoreIndex

AutoMerging_index = VectorStoreIndex(
    leaf_nodes,
    embed_model = OpenAIEmbedding(model='text-embedding-3-small'),
    storage_context=storage_context,
)

In [56]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [57]:
persist_path = "/content/drive/MyDrive/llamaindex/AutoMerging_index"
AutoMerging_index.storage_context.persist(persist_dir=persist_path)

In [58]:
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.embeddings.openai import OpenAIEmbedding

persist_path = "/content/drive/MyDrive/llamaindex/AutoMerging_index"
AutoMerging_storage_context = StorageContext.from_defaults(persist_dir=persist_path)

AutoMerging_Index_StorageContext = load_index_from_storage(
    AutoMerging_storage_context,
    embed_model=OpenAIEmbedding(model='text-embedding-3-small')
)

Loading llama_index.core.storage.kvstore.simple_kvstore from /content/drive/MyDrive/llamaindex/AutoMerging_index/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from /content/drive/MyDrive/llamaindex/AutoMerging_index/index_store.json.


In [59]:
AutoMerging_query_engine = AutoMerging_Index_StorageContext.as_query_engine()

In [60]:
AutoMerging_response = AutoMerging_query_engine.query(query)

In [61]:
Max_AutoMerging_score = 0.0

In [62]:
if hasattr(AutoMerging_response, 'source_nodes'):
    print("\nSource Nodes:")
    for i, node in enumerate(AutoMerging_response.source_nodes):
        text = node.node.text[:500]
        score = node.score if hasattr(node, 'score') else 'N/A'
        if score > Max_AutoMerging_score:
            Max_AutoMerging_score = score
        print(f"Node {i+1} (Score: {score:.4f}):\n{text}\n{'-'*80}\n")


Source Nodes:
Node 1 (Score: 0.5859):
Competitive factors impacting our business include, but are not limited to, pricing,
advertising, sales promotion programs,     in-store displays and point-of-sale
marketing, digital marketing, product and ingredient innovation, availability,
increased efficiency in production techniques, the introduction of new packaging as
well as new vending and dispensing equipment, contracting with marketing assets
(theaters, sports arenas, universities, etc.), and brand and trademark development
and protec
--------------------------------------------------------------------------------

Node 2 (Score: 0.5840):
Description of Business
The Coca-Cola Company is a total beverage company. We own or license and
market numerous beverage brands, which we group into the following categories:
Trademark Coca-Cola; sparkling flavors; water, sports, coffee and tea; juice, value-
added dairy and plant-based beverages; and emerging beverages.
------------------------------

In [63]:
print(Max_AutoMerging_score)

0.5859064740585999


In [64]:
print("Final Answer from AutoMerging:")
print(AutoMerging_response)

Final Answer from AutoMerging:
Coca-Cola's business model involves owning or licensing and marketing a variety of beverage brands grouped into categories such as Trademark Coca-Cola, sparkling flavors, water, sports, coffee and tea, juice, value-added dairy and plant-based beverages, and emerging beverages. They focus on factors like pricing, advertising, sales promotion programs, in-store displays, digital marketing, product innovation, efficiency in production techniques, packaging, vending equipment, marketing asset partnerships, and brand development and protection to impact their business competitively.


In [65]:
from IPython.display import Markdown, display

#response = query_engine.query("Tell me about Coca Cola's business model")
text = AutoMerging_response.response if hasattr(AutoMerging_response, "response") else str(AutoMerging_response)

display(Markdown(f"### Response from AutoMerging :\n\n{text}"))

### Response from AutoMerging :

Coca-Cola's business model involves owning or licensing and marketing a variety of beverage brands grouped into categories such as Trademark Coca-Cola, sparkling flavors, water, sports, coffee and tea, juice, value-added dairy and plant-based beverages, and emerging beverages. They focus on factors like pricing, advertising, sales promotion programs, in-store displays, digital marketing, product innovation, efficiency in production techniques, packaging, vending equipment, marketing asset partnerships, and brand development and protection to impact their business competitively.

## Auto retriever

In [66]:
%%capture
!pip install llama-index
!pip install llama-index-vector-stores-chroma

In [67]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore

In [68]:
from llama_index.core.schema import TextNode

nodes = [
    TextNode(
        text=(
            "The Coca-Cola Company is a total beverage company, and beverage products bearing our trademarks, sold in the United States since 1886, are now sold in more than 200 countries and territories."
            "We own or license and market numerous beverage brands, which we group into the following categories: Trademark Coca-Cola; sparkling flavors; water, sports, coffee and tea; juice, value-added dairy and plant-based beverages; and emerging beverages."
            "We own and market several of the world’s largest nonalcoholic sparkling soft drink brands, including Coca-Cola, Sprite, Coca-Cola Zero Sugar, Fanta and Diet Coke/Coca-Cola Light."
            "We make our branded beverage products available to consumers throughout the world through our network of independent bottling partners, distributors, wholesalers and retailers as well as our consolidated bottling and distribution operations. "
            "Beverages bearing trademarks owned by or licensed to the Company account for 2.2 billion of the estimated 65 billion servings of all beverages consumed worldwide every day."
            "We believe our success depends on our ability to connect with consumers by providing them with a wide variety of beverage options to meet their desires, needs and lifestyles."
            "Our success further depends on the ability of our people to execute effectively, every day."
            "We are guided by our purpose, which is to refresh the world and make a difference. Our vision for growth has three connected pillars:"
            "Loved Brands. We craft meaningful brands and a choice of drinks that people love and enjoy and that refresh them in body and spirit."
            "Done Sustainably. We grow our business in ways that achieve positive change in the world and build a more sustainable future for our planet."
            "For a Better Shared Future. We invest to improve people’s lives, from our employees to all those who touch our business system, to our investors, to the communities we call home."
            "The Coca-Cola Company was incorporated in September 1919 under the laws of the State of Delaware and succeeded to the business of a Georgia corporation with the same name that had been organized in 1892."
            "widely regarded as one of the greatest basketball players of all time."
        ),
        metadata={
            "category": "Business",
            "chapter": "Part I - ITEM 1",
        },
    ),
    TextNode(
        text=(
            "Our business, operating results, financial condition and liquidity may be adversely affected by changes in global economic conditions, including global inflationary pressures, prevailing interest rates, credit market conditions, increased unemployment, levels of consumer and business confidence, bank failures, commodity (including energy) prices and supply, a recession or economic slowdown, trade policies, foreign currency exchange rates, changing policy positions or priorities, governmental rules and approaches to taxation, levels of government spending and deficits, and actual or anticipated default on sovereign debt."
            "Many of the jurisdictions in which our products are sold have experienced, and could continue to experience, unfavorable changes in economic conditions, which could negatively affect the affordability of, and consumer demand for, our beverages, and certain markets in which our products are sold experienced intensified inflation throughout 2024, which may continue in 2025."
            "Under difficult economic conditions, consumers may seek to reduce discretionary spending by forgoing purchases of our products or by shifting away from our beverages to lower-priced products offered by other companies, including private-label brands, which could reduce our profitability and negatively affect our overall financial performance."
            "In addition, the occurrence of global or regional health events, and any related governmental, private sector and individual consumer responses, could contribute to a recession, depression or global economic downturn."
            "Other financial uncertainties in our major markets and unstable geopolitical conditions or events in certain markets, including international conflicts, civil unrest, acts of war, terrorism, governmental changes, or changes in international relations, could undermine global consumer confidence and reduce consumers’ purchasing power, thereby reducing demand for our products."
            "Throughout 2024, the Company faced disruption to our operations due to international conflicts, including the conflict between Russia and Ukraine and conflicts in the Middle East."
            "Geopolitical instability has in the past led, and may in the future lead, to logistical, transportation and supply chain disruptions; business disruptions (including labor shortages); increased risk of cybersecurity incidents or other disruptions to our information systems; reduced availability and increased costs of transportation, energy, packaging, raw materials and other input costs; and heightened security risk, impacting employee safety and/or damage to infrastructure or our assets."
            "At times, we have faced product boycotts resulting from political activism, which have reduced demand for our products. Restrictions on our ability to transfer earnings or capital across borders; price controls; limitations on profits; the negotiation of new trade agreements; new, expanded or retaliatory tariffs; import authorization requirements; and other restrictions on business activities, which have been or may be imposed or expanded as a result of political and economic instability, deterioration of economic relations between countries or otherwise, could impact our profitability."
            "In addition, U.S. trade sanctions against countries designated by the U.S. government as state sponsors of terrorism and/or financial institutions accepting transactions for commerce within such countries could increase significantly, which could make it difficult, or even impossible, for us to continue to make sales to bottlers in such countries. The imposition of retaliatory sanctions against U.S. multinational corporations by countries that are or may become subject to U.S. trade sanctions, or the delisting of our branded products by retailers in various countries in reaction to U.S."
            "trade sanctions or other governmental actions or policies, could also negatively affect our business."
        ),
        metadata={
            "category": "Risk Factors",
            "chapter": "Part I - ITEM 1A",
        },
    ),
    TextNode(
        text=(
            "Our worldwide headquarters is located on a 35-acre complex in Atlanta, Georgia."
            "The complex includes several office buildings which are used by Corporate employees and North America operating segment employees."
            "In addition, the complex includes technical and engineering facilities along with a reception center."
            "We own or lease additional facilities, real estate and office space throughout the world, which we use for administrative, manufacturing, processing, packaging, storage, warehousing, distribution and retail operations."
            "These properties are generally included in the geographic operating segment in which they are located, with the exception of our Costa retail stores, which are included in the Global Ventures operating segment, and facilities related to our consolidated bottling and distribution operations, which are included in the Bottling Investments operating segment."
            "Management believes that our Company’s facilities used for the production of our products are suitable and adequate, that they are being appropriately utilized in line with past experience, and that they have sufficient production capacity for their present intended purposes."
            "The extent of utilization of our production facilities varies based upon seasonal demand for our products."
            "However, management believes that, with the exception of certain dairy products that require specialized equipment, additional production can be achieved at the existing facilities by adding personnel and capital equipment or, at some facilities, by adding shifts of personnel or expanding the facilities. The Company is in the process of increasing our dairy production capacity."
            "We continuously review our anticipated requirements for facilities and, on the basis of that review, may from time to time acquire or lease additional facilities and/or dispose of existing facilities."

        ),
        metadata={
            "category": "Properties",
            "chapter": "Part I - ITEM 2",
        },
    ),
    TextNode(
        text=(
            "During the fiscal quarter ended December 31, 2024, none of our Directors or officers (as defined in Rule 16a-1(f) under the Exchange Act) adopted or terminated a “Rule 10b5-1 trading arrangement” or a “non-Rule 10b5-1 trading arrangement,” as each term is defined in Item 408 of Regulation S-K, except as follows:"
            "Jennifer K. Mann, Executive Vice President and President, North America operating unit, adopted a Rule 10b5-1 trading arrangement on November 22, 2024 for the potential exercise of vested stock options and the associated sale of up to 80,820 shares of common stock of the Company, subject to certain conditions."
            "The arrangement’s expiration date is November 1, 2025, or such earlier date upon which all transactions are completed."
            "This trading plan was adopted during an open trading window."
        ),
        metadata={
            "category": "Other Information",
            "chapter": "Part II - ITEM 9B",
        },
    ),

]

In [69]:
%%capture
!pip install chromadb

In [70]:
import chromadb
chroma_client = chromadb.EphemeralClient()
chroma_collection = chroma_client.get_or_create_collection("quickstart")


In [71]:
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [72]:
Auto_index = VectorStoreIndex(nodes, storage_context=storage_context)

In [73]:
Auto_query_engine = Auto_index.as_query_engine()

In [74]:
Auto_response = Auto_query_engine.query(query)

In [75]:
Max_Auto_score = 0.0

In [76]:
if hasattr(Auto_response, 'source_nodes'):
    print("\nSource Nodes:")
    for i, node in enumerate(Auto_response.source_nodes):
        text = node.node.text[:500]
        score = node.score if hasattr(node, 'score') else 'N/A'
        if score > Max_Auto_score:
            Max_Auto_score = score
        print(f"Node {i+1} (Score: {score:.4f}):\n{text}\n{'-'*80}\n")


Source Nodes:
Node 1 (Score: 0.7489):
The Coca-Cola Company is a total beverage company, and beverage products bearing our trademarks, sold in the United States since 1886, are now sold in more than 200 countries and territories.We own or license and market numerous beverage brands, which we group into the following categories: Trademark Coca-Cola; sparkling flavors; water, sports, coffee and tea; juice, value-added dairy and plant-based beverages; and emerging beverages.We own and market several of the world’s largest nonalcoholic 
--------------------------------------------------------------------------------

Node 2 (Score: 0.6471):
Our worldwide headquarters is located on a 35-acre complex in Atlanta, Georgia.The complex includes several office buildings which are used by Corporate employees and North America operating segment employees.In addition, the complex includes technical and engineering facilities along with a reception center.We own or lease additional facilities, real 

In [77]:
print(Max_Auto_score)

0.7489465773135379


In [78]:
print("Final Answer from Auto:")
print(Auto_response)

Final Answer from Auto:
Coca-Cola operates as a total beverage company with a wide range of beverage products sold globally. The company owns or licenses various beverage brands grouped into categories such as Trademark Coca-Cola, sparkling flavors, water, sports, coffee and tea, juice, value-added dairy, plant-based beverages, and emerging beverages. Coca-Cola markets some of the world's largest nonalcoholic sparkling soft drink brands like Coca-Cola, Sprite, Fanta, and Diet Coke. The company distributes its branded beverage products worldwide through a network of independent bottling partners, distributors, wholesalers, and retailers. Coca-Cola's success is attributed to its ability to offer consumers a diverse range of beverage options to meet their preferences and lifestyles. The company's vision for growth is centered on crafting loved brands, sustainable growth practices, and investing in a better shared future.


In [79]:
from IPython.display import Markdown, display

#response = query_engine.query("Tell me about Coca Cola's business model")
text = Auto_response.response if hasattr(Auto_response, "response") else str(Auto_response)

display(Markdown(f"### Response from Auto :\n\n{text}"))

### Response from Auto :

Coca-Cola operates as a total beverage company with a wide range of beverage products sold globally. The company owns or licenses various beverage brands grouped into categories such as Trademark Coca-Cola, sparkling flavors, water, sports, coffee and tea, juice, value-added dairy, plant-based beverages, and emerging beverages. Coca-Cola markets some of the world's largest nonalcoholic sparkling soft drink brands like Coca-Cola, Sprite, Fanta, and Diet Coke. The company distributes its branded beverage products worldwide through a network of independent bottling partners, distributors, wholesalers, and retailers. Coca-Cola's success is attributed to its ability to offer consumers a diverse range of beverage options to meet their preferences and lifestyles. The company's vision for growth is centered on crafting loved brands, sustainable growth practices, and investing in a better shared future.

## Retrieval evaluation conclusion

In [80]:
print(f"""Retriever score comparison with SentenceWindowRetrieval, AutoMerging, and Auto retrieval is given below.

 Sentance Window Retriever: {Max_SentenceWindow_score}
 Auto Merging Retriever: {Max_AutoMerging_score}
 Auto Retriever: {Max_Auto_score}

 Sentance Window Retriever is showing better perfromance than Auto Merging and Auto Sentance Window Retriever.
""")

Retriever score comparison with SentenceWindowRetrieval, AutoMerging, and Auto retrieval is given below.

 Sentance Window Retriever: 0.7620215300783083
 Auto Merging Retriever: 0.5859064740585999
 Auto Retriever: 0.7489465773135379

 Sentance Window Retriever is showing better perfromance than Auto Merging and Auto Sentance Window Retriever.



## Router Query Engine

Router query engine enables efficient quering and data retrieval across multiple sources or services. </br>

I have implemented router query engine by creating two index (vector & summary), followed by creating two query engine (vector & summary) based on the index respectively. Two tools vector and summary are defined that the router engine can choose from based on user query.

</br>

Behaviour of the router query is explained in conclusion section.




In [81]:
import nest_asyncio

nest_asyncio.apply()

In [82]:
%%capture
!pip install llama-index

In [83]:
import logging
import sys

# Set up the root logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)  # Set logger level to INFO

# Clear out any existing handlers
logger.handlers = []

# Set up the StreamHandler to output to sys.stdout (Colab's output)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)  # Set handler level to INFO

# Add the handler to the logger
logger.addHandler(handler)


In [84]:
from llama_index.core import VectorStoreIndex, SummaryIndex, SimpleDirectoryReader, StorageContext
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from IPython.display import display, HTML

In [85]:
import os
from dotenv import load_dotenv, find_dotenv

In [86]:
cwd = os.getcwd()
print("Current working directory:", cwd)

Current working directory: /content/data


In [87]:
%%capture
!mkdir data
!wget "https://investors.coca-colacompany.com/filings-reports/all-sec-filings/content/0000021344-25-000011/ko-20241231.htm" -O 'data/CocaCola_annual_10K_data.pdf'


In [88]:
import os
print(os.getcwd())
os.chdir('./data')
print(os.getcwd())
os.listdir

/content/data
/content/data/data


<function posix.listdir(path=None)>

In [91]:
Document_2025 = SimpleDirectoryReader(input_files=['/The_Coca_Cola_Company_10K_Feb_2025.pdf']).load_data()

In [92]:
nodes_2025 = Settings.node_parser.get_nodes_from_documents(Document_2025, chunk_size=1024)

## Define Summary/Vector Index, Query Engine, and Tools

In [93]:
llm = OpenAI(model="gpt-3.5-turbo")

In [94]:
from llama_index.embeddings.openai import OpenAIEmbedding

embed_model = OpenAIEmbedding(model="text-embedding-3-small")

summary_index_2025 = SummaryIndex(nodes_2025, embed_model=embed_model,llm=llm)

vector_index_2025 = VectorStoreIndex(nodes_2025, embed_model=embed_model)

HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [95]:
summary_query_engine_2025 = summary_index_2025.as_query_engine(response_mode="tree_summarize",use_async=True,)

vector_query_engine_2025 = vector_index_2025.as_query_engine()

In [96]:
from llama_index.core.tools.query_engine import QueryEngineTool

# Summary Index tool
summary_tool_2025 = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine_2025,
    description="Useful for summarization of Coca Cola 10K.",
)

# Vector Index tool
vector_tool_2025 = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine_2025,
    description="Useful for retrieving specific context from Coca Cola 10K.",
)

## Define Router Query Engine

In [97]:
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors.llm_selectors import LLMSingleSelector, LLMMultiSelector
from llama_index.core.selectors.pydantic_selectors import PydanticMultiSelector, PydanticSingleSelector

In [98]:
# Create Router Query Engine
router_query_engine_2025 = RouterQueryEngine(
    selector=PydanticSingleSelector.from_defaults(),
    query_engine_tools=[
        summary_tool_2025,
        vector_tool_2025,
    ],
)

In [99]:
response = router_query_engine_2025.query("Give me summary of this documents in 10 bullet points?")

HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Selecting query engine 0: The choice 'Useful for summarization of Coca Cola 10K' is most relevant to the question as it specifically mentions summarization, which aligns with the request for a summary of the document in 10 bullet points..
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com

In [100]:
print(response)

- The Coca-Cola Company's annual report for the fiscal year ended December 31, 2024, covers operating segments, product portfolio, distribution system, and competitive landscape.
- The report discusses raw materials, government regulations, human capital management strategies, and global talent development programs.
- Key priorities for the Company include talent retention, leadership development, fostering an inclusive culture, and successful negotiations of collective bargaining agreements.
- Legal compliance, risk management, and maintaining consumer confidence in product safety and quality are critical for the Company.
- Challenges faced by the Company include obesity concerns, evolving consumer preferences, competition in the digital marketplace, and talent acquisition and retention.
- Critical accounting policies and estimates, consolidation principles, revenue recognition, and income taxes are detailed in the financial statements.
- The Company uses derivative financial instrume

In [ ]:
display(HTML(f'<p style="font-size:14px">{response.response}</p>'))

In [101]:
response = router_query_engine_2025.query("Tell me about Coca Cola leage issues?")

HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Selecting query engine 1: The question 'Tell me about Coca Cola leage issues?' requires retrieving specific context from Coca Cola 10K, which aligns with choice (2).
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [102]:
display(HTML(f'<p style="font-size:14px">{response.response}</p>'))

## Conclusion of router query engine </br>

When "Give me summary of this documents in 10 bullet points?" is prompted, its routed to summary tool and the response generated is a summary with  10 bullet points. </br>

</br>
When "Tell me about Coca Cola leage issues?" is prompted, its routed to summary vector and the generate response associated with leagel issues of Coca Cola. </br>

## Sub question query engine

In [103]:
%%capture
!pip install llama-index

In [104]:
from llama_index.core import SimpleDirectoryReader

In [105]:
Document_2014 = SimpleDirectoryReader(input_files=['/The_Coca_Cola_Company_10K_Feb_2014.pdf']).load_data()

In [106]:
from llama_index.core.settings import Settings
from llama_index.core import VectorStoreIndex

In [107]:
nodes_2014 = Settings.node_parser.get_nodes_from_documents(Document_2014, chunk_size=1024)

In [108]:
vector_index_2014 = VectorStoreIndex(nodes_2014, embed_model=embed_model,llm=llm)

HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [109]:
vector_query_engine_2014 = vector_index_2014.as_query_engine()

In [ ]:
#Document_2014_index = VectorStoreIndex.from_documents(Simple_Documents_2014)
#query_engine_2014 = Document_2014_index.as_query_engine()

In [120]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata

In [121]:
tool_2025 = QueryEngineTool(
    query_engine=vector_query_engine_2025,
    metadata={
        "name": "Coca_Cola_2025",
        "description": "Answers questions from Coca Cola 2025"
    }
)

tool_2014 = QueryEngineTool(
    query_engine=vector_query_engine_2014,
    metadata={
        "name": "Coca_Cola_2014",
        "description": "Answers questions from Coca Cola 2014"
    }
)

In [122]:
tools = [tool_2025, tool_2014]

In [123]:
tools[0].metadata

{'name': 'Coca_Cola_2025',
 'description': 'Answers questions from Coca Cola 2025'}

In [124]:
from llama_index.core.query_engine import SubQuestionQueryEngine

In [125]:
sub_query_engine = SubQuestionQueryEngine.from_defaults(query_engine_tools=tools)

AttributeError: 'dict' object has no attribute 'name'

In [127]:
tools = [
    QueryEngineTool(
        query_engine=vector_query_engine_2025,
        metadata=ToolMetadata(
            name="Coca_Cola_2025",
            description="Answers questions from Coca Cola 2025"
        )
    ),
    QueryEngineTool(
        query_engine=vector_query_engine_2014,
        metadata=ToolMetadata(
            name="Coca_Cola_2014",
            description="Answers questions from Coca Cola 2014"
        )
    )
]

In [128]:
sub_query_engine = SubQuestionQueryEngine.from_defaults(query_engine_tools=tools)

In [137]:
query = "Give comprehensive detail about Coca Cola's risk factors in 2014 and comprehensive detail about Coca Cola's risk factors in 2025?"
response = sub_query_engine.query(query)
print(response)

HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generated 2 sub questions.
[Coca_Cola_2014] Q: What were Coca Cola's risk factors in 2014?
Retrying request to /embeddings in 0.456813 seconds
[Coca_Cola_2025] Q: What were Coca Cola's risk factors in 2025?
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Retrying request to /chat/completions in 0.394051 seconds
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[Coca_Cola_2025] A: Coca-Cola's risk factors in 2025 included challenges related to obesity, increased demand for food products affecting agricultural productivity, regulation of ingredient sourcing due diligence, climate change, and legal or regulatory responses to climate change.
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[Coca_Cola_2014] A: Coca Cola's risk factors in 2014 included co

In [142]:

display(Markdown(f"### Response:\n\n{response}"))

### Response:

Coca Cola's risk factors in 2014 included concerns about obesity potentially reducing demand for their products, water scarcity and poor quality impacting production costs, evolving consumer preferences requiring more choices, increased competition in the marketplace, ensuring product safety and quality, and potential food security issues due to changing weather patterns affecting agricultural productivity. In 2025, Coca-Cola's risk factors included challenges related to obesity, increased demand for food products affecting agricultural productivity, regulation of ingredient sourcing due diligence, climate change, and legal or regulatory responses to climate change.

In [144]:
if "2014" in str(response) and "2025" in str(response):
    # Basic split using years
    response_text = str(response)
    parts = response_text.split("2025")

    # Attempt to segment based on appearance of years
    part_2014 = parts[0].split("2014")[-1].strip()
    part_2025 = parts[1].strip() if len(parts) > 1 else ""

    print("🗓️  **2014 Risk Factors**\n")
    #print(part_2014 if part_2014 else "No details found.")
    display(Markdown(f"### Response:\n\n{part_2014}"))
    print("\n" + "-"*80 + "\n")
    print("🗓️  **2025 Risk Factors**\n")
    #print(part_2025 if part_2025 else "No details found.")
    display(Markdown(f"### Response:\n\n{part_2025}"))
else:
    # Fallback: Just show raw output if formatting is unreliable
    print(str(response))

🗓️  **2014 Risk Factors**



### Response:

included concerns about obesity potentially reducing demand for their products, water scarcity and poor quality impacting production costs, evolving consumer preferences requiring more choices, increased competition in the marketplace, ensuring product safety and quality, and potential food security issues due to changing weather patterns affecting agricultural productivity. In


--------------------------------------------------------------------------------

🗓️  **2025 Risk Factors**



### Response:

, Coca-Cola's risk factors included challenges related to obesity, increased demand for food products affecting agricultural productivity, regulation of ingredient sourcing due diligence, climate change, and legal or regulatory responses to climate change.

In [ ]:
Legal Proceedings

In [145]:
query = "Give comprehensive detail about Coca Cola's Legal Proceedings in 2014 and comprehensive detail about Coca Cola's Legal Proceedings in 2025?"
response = sub_query_engine.query(query)
print(response)

HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generated 2 sub questions.
[Coca_Cola_2014] Q: What were the legal proceedings of Coca Cola in 2014?
Retrying request to /embeddings in 0.450840 seconds
[Coca_Cola_2025] Q: What were the legal proceedings of Coca Cola in 2025?
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Retrying request to /chat/completions in 0.375911 seconds
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[Coca_Cola_2014] A: Coca-Cola was involved in legal proceedings with Aqua-Chem, where they believed they had substantial legal and factual defenses to Aqua-Chem's claims. The parties entered into litigation in Georgia, which was stayed pending the outcome of litigation filed in Wisconsin by certain insurers of Aqua-Chem. Settlements were reached with several insurers, who paid funds into an escrow accoun

In [146]:
if "2014" in str(response) and "2025" in str(response):
    # Basic split using years
    response_text = str(response)
    parts = response_text.split("2025")

    # Attempt to segment based on appearance of years
    part_2014 = parts[0].split("2014")[-1].strip()
    part_2025 = parts[1].strip() if len(parts) > 1 else ""

    print("🗓️  **2014 Risk Factors**\n")
    #print(part_2014 if part_2014 else "No details found.")
    display(Markdown(f"### Response:\n\n{part_2014}"))
    print("\n" + "-"*80 + "\n")
    print("🗓️  **2025 Risk Factors**\n")
    #print(part_2025 if part_2025 else "No details found.")
    display(Markdown(f"### Response:\n\n{part_2025}"))
else:
    # Fallback: Just show raw output if formatting is unreliable
    print(str(response))

🗓️  **2014 Risk Factors**



### Response:

, Coca-Cola was involved in legal proceedings with Aqua-Chem, where they believed they had substantial legal and factual defenses to Aqua-Chem's claims. The parties entered into litigation in Georgia, which was stayed pending the outcome of litigation filed in Wisconsin by certain insurers of Aqua-Chem. Settlements were reached with several insurers, who paid funds into an escrow account for costs arising from asbestos claims against Aqua-Chem.

In


--------------------------------------------------------------------------------

🗓️  **2025 Risk Factors**



### Response:

, Coca-Cola faced legal proceedings related to environmental impacts of plastic packaging in Baltimore and Los Angeles. The lawsuit from Baltimore involved claims of violations of state statutes, unfair trade practices, design defects, and negligence. The Los Angeles lawsuit included claims of public nuisance, violations of California laws, and false advertising. Coca-Cola believes it has strong defenses against these claims.